# DEC5 000973: local seam and primary-occlusion mixtures
Two controls × two RGB modes × three held camera anchors = 12 outputs at one temporal frame. All complete native gates fail. These checks preserve paired input and metric definitions; a small face-metric gain cannot certify the neck/hand outside the ROI.


In [1]:
from pathlib import Path
import hashlib, json
import numpy as np
from PIL import Image
root = Path('/mnt/data/lookcloser_dec5_5a3_surface_repair')
families = ['seam_local_source_control', 'primary_fallback_source_control']
def sha(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024*1024), b''):
            digest.update(block)
    return digest.hexdigest()
all_rows = []
for family in families:
    result = json.loads((root/family/'findings.json').read_text())
    assert result['audit_status'] == 'comparison_integrity_pass_candidate_quality_fail'
    assert result['temporal_frame_count'] == 1 and result['variant_anchor_count'] == 6
    assert result['visual_pass'] == 0 and result['visual_fail'] == 6
    assert len({(r['anchor'],r['variant']) for r in result['rows']}) == 6
    for path, digest in result['retained_hashes'].items():
        assert sha(path) == digest, path
    all_rows.extend(dict(family=family, **r) for r in result['rows'])
    print(family, 'hashes verified:', len(result['retained_hashes']))
assert len(all_rows) == 12


seam_local_source_control hashes verified: 163
primary_fallback_source_control hashes verified: 164


In [2]:
baseline = json.loads((root/'base_transport_control/F/metrics_baseline/metrics.json').read_text())
for family in families:
    for variant in ['full_rgb','low_band']:
        m = json.loads((root/family/'F'/('metrics_'+variant)/'metrics.json').read_text())
        for key in ['ground_truth_sha256','face_polygons_sha256','face_bbox_xyxy','protocol','metric_domain']:
            assert m[key] == baseline[key]
        assert all(np.isfinite(m[k]) for k in ['face_psnr','face_ssim','face_lpips'])
        assert not any(k.startswith(('loss','full_frame','room_psnr','actor_psnr')) for k in m)
        print(family, variant, {k:m[k] for k in ['face_psnr','face_ssim','face_lpips']})
print('Identical study ROI:', baseline['face_bbox_xyxy'], '; most neck/hand lies outside it.')


seam_local_source_control full_rgb {'face_psnr': 28.784076690673828, 'face_ssim': 0.8905367851257324, 'face_lpips': 0.0483241081237793}
seam_local_source_control low_band {'face_psnr': 28.7495059967041, 'face_ssim': 0.8891779184341431, 'face_lpips': 0.04766947403550148}
primary_fallback_source_control full_rgb {'face_psnr': 28.81765365600586, 'face_ssim': 0.8913882374763489, 'face_lpips': 0.05010417103767395}
primary_fallback_source_control low_band {'face_psnr': 28.78635597229004, 'face_ssim': 0.8892638683319092, 'face_lpips': 0.04743494838476181}
Identical study ROI: [700, 388, 1165, 726] ; most neck/hand lies outside it.


In [3]:
for family in families:
    for anchor in ['F','J','L']:
        directory = root/family/anchor
        request = json.loads((directory/'request.json').read_text())
        assert request['source_rgb_averaging'] and not request['uses_eval_rgb'] and not request['uses_semantic_masks']
        assert request['train_camera_count'] == 62 and len(set(request['sources'])) == 8
        warps = next(Path(p).parent for p in request['input_hashes'] if Path(p).name == 'source_00.png')
        valid = np.stack([np.asarray(Image.open(warps/f'valid_{i:02d}.png'))>0 for i in range(8)])
        weights = np.load(directory/'source_weights.npz',allow_pickle=False)['weights']
        band = np.asarray(Image.open(directory/'seam_band.png'))>0
        assert weights.shape == (8,1080,1920) and np.isfinite(weights).all()
        assert (weights>=0).all() and not weights[~valid].any()
        total=weights.sum(0)
        assert np.max(np.abs(total[total>0]-1))<3e-7
        mixed=(weights>0).sum(0)>1
        assert not (mixed&~band).any()
        unchanged=~mixed if family=='seam_local_source_control' else ~band
        base=np.asarray(Image.open(directory/'baseline_pred_0000.png'))
        for variant in ['full_rgb','low_band']:
            pred=np.asarray(Image.open(directory/variant/'eval_pred_0000.png'))
            assert np.array_equal(pred[unchanged],base[unchanged])
        print(family,anchor,'unchanged PNG pixels:',int(unchanged.sum()))


seam_local_source_control F unchanged PNG pixels: 1813200


seam_local_source_control J unchanged PNG pixels: 1819553


seam_local_source_control L unchanged PNG pixels: 1801282


primary_fallback_source_control F unchanged PNG pixels: 1820745


primary_fallback_source_control J unchanged PNG pixels: 1800414


primary_fallback_source_control L unchanged PNG pixels: 1764463


In [4]:
point_path=root/'primary_fallback_source_control/F/neck_point_audit.json'
evidence=json.loads(point_path.read_text())
assert evidence['posthoc_only'] and not evidence['gt_used_for_predictor']
for path,digest in evidence['input_hashes'].items():
    assert sha(path)==digest
point=next(r for r in evidence['rows'] if r['pixel']==[580,660])
assert point['visible_ranks']==[2,3,4,5,7]
assert all(point['weights'][i]>0 for i in point['visible_ranks'])
assert point['gt_outside_channel_range'][:2]==[True,True]
print(point)
print('Scope: five visible sources from the frozen eight-source pool at one diagnostic point, not all 62 cameras.')


{'baseline_rgb8': [168, 125, 87], 'fallback_rgb8': [169, 124, 87], 'gt_display_rgb255': [160.0, 117.0, 83.0], 'gt_outside_channel_range': [True, True, False], 'pixel': [580, 660], 'source_rgb8_max': [171, 127, 91], 'source_rgb8_min': [165, 119, 80], 'visible_ranks': [2, 3, 4, 5, 7], 'weights': [0.0, 0.0, 0.2984232008457184, 0.2492876946926117, 0.17633923888206482, 0.1679423600435257, 0.0, 0.10800749808549881]}
Scope: five visible sources from the frozen eight-source pool at one diagnostic point, not all 62 cameras.


## Interpretation
Both scoped mixtures avoid global face blurring but leave the conspicuous neck/tube problem. The sampled fallback RGBs are similar, so mixing them barely changes the interior; some angular tube-adjacent patches become more visible. This rejects these controls as complete repair, not the overall goal. Native texture/focus/noise mismatch and false surface projection need to be separated before further weight tuning. No temporal/fly-through promotion.
